In [27]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Core libraries
import pandas as pd
import numpy as np

# Visualization (optional)
import matplotlib.pyplot as plt
import seaborn as sns

# ML & preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Imbalance handling
from imblearn.over_sampling import SMOTE


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
# Dataset path
data_path = '/content/drive/MyDrive/credit_fraud_detect/creditcard.csv'

df = pd.read_csv(data_path)
df.head()


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [29]:
print(df.shape)
print(df['Class'].value_counts())


(284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64


In [30]:
# Features & target
X = df.drop('Class', axis=1)
y = df['Class']

# Scale Amount column (important)
scaler = StandardScaler()
X['Amount'] = scaler.fit_transform(X[['Amount']])

# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## 🟦   **Logistic Regression**

In [31]:

# ----------------------------
# Feature scaling (ALL features)
# ----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

# ----------------------------
# SMOTE
# ----------------------------
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# ----------------------------
# Logistic Regression (Fixed)
# ----------------------------
lr = LogisticRegression(
    solver='liblinear',
    max_iter=5000,
    random_state=42
)

lr.fit(X_train_smote, y_train_smote)

# Predictions
y_pred_lr = lr.predict(X_test)

# Evaluation
print("Corrected Logistic Regression Results")
print(classification_report(y_test, y_pred_lr))

print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1 Score:", f1_score(y_test, y_pred_lr))


Corrected Logistic Regression Results
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.97     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.97      0.99     56962

Precision: 0.05787781350482315
Recall: 0.9183673469387755
F1 Score: 0.1088929219600726


In [32]:
results = {
    "model": "Logistic Regression",
    "precision": precision_score(y_test, y_pred_lr),
    "recall": recall_score(y_test, y_pred_lr),
    "f1_score": f1_score(y_test, y_pred_lr)
}

results

{'model': 'Logistic Regression',
 'precision': 0.05787781350482315,
 'recall': 0.9183673469387755,
 'f1_score': 0.1088929219600726}

###  Logistic Regression achieves very high recall but poor precision due to extreme class imbalance and its linear nature. While effective at detecting most fraud cases, it results in a high false positive rate, making it less suitable for production use in fraud detection systems. This motivates the use of ensemble models such as Random Forest and XGBoost, which capture non-linear relationships more effectively.

## 🟦   **Random Forest**

### ⏩  **Random Forest With SMOTE**


In [19]:
# ----------------------------
# Apply SMOTE
# ----------------------------
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Class distribution after SMOTE:")
print(y_train_smote.value_counts())

# ----------------------------
# Random Forest WITH SMOTE
# ----------------------------
rf_smote = RandomForestClassifier(
    n_estimators=100,          # keep smaller to reduce time
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)

rf_smote.fit(X_train_smote, y_train_smote)

y_pred_rf_smote = rf_smote.predict(X_test)

print("Random Forest WITH SMOTE Results")
print(classification_report(y_test, y_pred_rf_smote))

print("Precision:", precision_score(y_test, y_pred_rf_smote))
print("Recall:", recall_score(y_test, y_pred_rf_smote))
print("F1 Score:", f1_score(y_test, y_pred_rf_smote))


Class distribution after SMOTE:
Class
0    227451
1    227451
Name: count, dtype: int64
Random Forest WITH SMOTE Results
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.56      0.85      0.67        98

    accuracy                           1.00     56962
   macro avg       0.78      0.92      0.84     56962
weighted avg       1.00      1.00      1.00     56962

Precision: 0.5570469798657718
Recall: 0.8469387755102041
F1 Score: 0.6720647773279352


In [21]:
results = {
    "model": "Random Forest",
    "precision": precision_score(y_test, y_pred_rf_smote),
    "recall": recall_score(y_test, y_pred_rf_smote),
    "f1_score": f1_score(y_test, y_pred_rf_smote)
}

results


{'model': 'Random Forest',
 'precision': 0.5570469798657718,
 'recall': 0.8469387755102041,
 'f1_score': 0.6720647773279352}

### ⏩   **Random Forest Without SMOTE**


In [18]:
rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Optimized Random Forest Results")
print(classification_report(y_test, y_pred_rf))

print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))


Optimized Random Forest Results
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.84      0.84      0.84        98

    accuracy                           1.00     56962
   macro avg       0.92      0.92      0.92     56962
weighted avg       1.00      1.00      1.00     56962

Precision: 0.8367346938775511
Recall: 0.8367346938775511
F1 Score: 0.8367346938775511


In [20]:
results = {
    "model": "Random Forest",
    "precision": precision_score(y_test, y_pred_rf),
    "recall": recall_score(y_test, y_pred_rf),
    "f1_score": f1_score(y_test, y_pred_rf)
}

results


{'model': 'Random Forest',
 'precision': 0.8367346938775511,
 'recall': 0.8367346938775511,
 'f1_score': 0.8367346938775511}

### Random Forest combined with SMOTE significantly increased training time due to a large synthetic dataset. Therefore, class weighting was used instead of oversampling, resulting in faster training while preserving model performance.

## 🟦   **XGBOOST**

### ⏩   **XGBOOST With SMOTE**

In [33]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_smote.value_counts())


pos_weight = len(y_train_smote[y_train_smote==0]) / len(y_train_smote[y_train_smote==1])

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    scale_pos_weight=pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train_smote, y_train_smote)
y_pred_xgb = xgb.predict(X_test)

print(classification_report(y_test, y_pred_xgb))


print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb))


Before SMOTE: Class
0    227451
1       394
Name: count, dtype: int64
After SMOTE: Class
0    227451
1    227451
Name: count, dtype: int64
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.58      0.87      0.70        98

    accuracy                           1.00     56962
   macro avg       0.79      0.93      0.85     56962
weighted avg       1.00      1.00      1.00     56962

Precision: 0.5821917808219178
Recall: 0.8673469387755102
F1 Score: 0.6967213114754098


In [34]:
results = {
    "model": "XGBoost",
    "precision": precision_score(y_test, y_pred_xgb),
    "recall": recall_score(y_test, y_pred_xgb),
    "f1_score": f1_score(y_test, y_pred_xgb)
}

results


{'model': 'XGBoost',
 'precision': 0.5821917808219178,
 'recall': 0.8673469387755102,
 'f1_score': 0.6967213114754098}

### ⏩   **XGBOOST Without SMOTE**

In [24]:
# --------------------------------
# Calculate scale_pos_weight
# --------------------------------
pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

# --------------------------------
# XGBoost model (NO SMOTE)
# --------------------------------
xgb_no_smote = XGBClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=3,
    gamma=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    random_state=42,
    eval_metric='logloss'
)

# Train on ORIGINAL imbalanced data
xgb_no_smote.fit(X_train, y_train)

# Predictions
y_pred_xgb_no_smote = xgb_no_smote.predict(X_test)

# Evaluation
print("XGBoost WITHOUT SMOTE Results")
print(classification_report(y_test, y_pred_xgb_no_smote))

print("Precision:", precision_score(y_test, y_pred_xgb_no_smote))
print("Recall:", recall_score(y_test, y_pred_xgb_no_smote))
print("F1 Score:", f1_score(y_test, y_pred_xgb_no_smote))


XGBoost WITHOUT SMOTE Results
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.82      0.84      0.83        98

    accuracy                           1.00     56962
   macro avg       0.91      0.92      0.91     56962
weighted avg       1.00      1.00      1.00     56962

Precision: 0.82
Recall: 0.8367346938775511
F1 Score: 0.8282828282828283


In [25]:
results = {
    "model": "XGBoost",
    "precision": precision_score(y_test, y_pred_xgb_no_smote),
    "recall": recall_score(y_test, y_pred_xgb_no_smote),
    "f1_score": f1_score(y_test, y_pred_xgb_no_smote)
}

results


{'model': 'XGBoost',
 'precision': 0.82,
 'recall': 0.8367346938775511,
 'f1_score': 0.8282828282828283}

###When XGBoost was trained using SMOTE, the model achieved higher recall but suffered from reduced precision due to the introduction of synthetic minority samples, leading to more false positives. In contrast, training XGBoost without SMOTE and using class weighting resulted in significantly higher precision while maintaining comparable recall. Overall, XGBoost without SMOTE demonstrated better balance and generalization, making it the preferred and more reliable model for fraud detection.